# 08 — Cross-encoder Reranking
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:**
- `data/processed/books_with_emotions.csv`
- `data/chroma_db/` (BGE-small embeddings)

**Output:**
- Two-stage retrieval pipeline using Cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) to rerank candidates retrieved by Bi-encoder

## 0. Setup

In [1]:
# pip install sentence-transformers chromadb
import pandas as pd
import numpy as np
import pickle, json, time
import scipy.sparse as sp
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import chromadb
from chromadb.config import Settings

DATA_PATH   = Path('data/processed/books_with_emotions.csv')
MODEL_PATH  = Path('models')
CHROMA_PATH = Path('data/chroma_db')
EVAL_PATH   = Path('data/eval/test_queries.json')
REPORT_PATH = Path('reports')
FIGURE_PATH = Path('reports/figures')
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150,
    'figure.facecolor': 'white', 'axes.facecolor': '#F9F9F9',
    'axes.spines.top': False, 'axes.spines.right': False,
})
MODEL_COLORS = {
    'TF-IDF'   : '#6B8CBA',
    'BM25'     : '#F4A261',
    'Semantic' : '#2A9D8F',
    'Hybrid'   : '#E76F51',
    'Reranking': '#8338EC',
}
print('Setup OK')

Setup OK


## 1. Load Data & Models

In [2]:
df = pd.read_csv(DATA_PATH)
df['isbn13'] = df['isbn13'].astype(str)
print(f'Books: {len(df):,}')

Books: 11,606


In [3]:
# --- Stage 1: BGE-small bi-encoder ---
print('Loading BGE-small...')
bi_encoder = SentenceTransformer('BAAI/bge-small-en-v1.5')
BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
    settings=Settings(anonymized_telemetry=False),
)
collection = chroma_client.get_collection('books')
print(f'ChromaDB count: {collection.count():,}')

# --- Stage 2: Cross-encoder ---
CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
print(f'Loading cross-encoder: {CROSS_ENCODER_MODEL}')
t0 = time.time()
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, max_length=512)
print(f'Cross-encoder loaded in {time.time()-t0:.1f}s')

# --- BM25 (for hybrid option in first stage) ---
print('Loading BM25...')
with open(MODEL_PATH / 'bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)

print('All models loaded ✓')

Loading BGE-small...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ChromaDB count: 11,606
Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder loaded in 8.5s
Loading BM25...
All models loaded ✓


## 2. Search Functions

In [4]:
def search_semantic(query, top_k=10):
    q_emb = bi_encoder.encode([BGE_PREFIX + query], normalize_embeddings=True)
    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['metadatas', 'distances', 'documents'],
    )
    rows = [{
        'isbn13'     : results['ids'][0][i],
        'title'      : results['metadatas'][0][i].get('title', ''),
        'authors'    : results['metadatas'][0][i].get('authors', ''),
        'categories' : results['metadatas'][0][i].get('categories', ''),
        'description': results['documents'][0][i],
        'score'      : round(1 - results['distances'][0][i], 4),
    } for i in range(len(results['ids'][0]))]
    return pd.DataFrame(rows)

print('Base semantic search ready')

Base semantic search ready


## 3. Cross-encoder Reranking Pipeline

In [5]:
def rerank_with_cross_encoder(
    query: str,
    candidates: pd.DataFrame,
    top_k: int = 5,
) -> pd.DataFrame:
    """
    Rerank candidate documents using a cross-encoder.

    Args:
        query      : original search query
        candidates : DataFrame with 'description' column
        top_k      : number of results to return after reranking

    Returns:
        Top-k reranked DataFrame with 'rerank_score' column
    """
    if candidates.empty:
        return candidates

    # Build (query, passage) pairs
    pairs = [(query, desc) for desc in candidates['description']]

    # Cross-encoder scores — higher = more relevant
    scores = cross_encoder.predict(pairs, show_progress_bar=False)

    result = candidates.copy()
    result['rerank_score'] = scores
    result = result.sort_values('rerank_score', ascending=False).head(top_k)
    return result.reset_index(drop=True)


def search_with_reranking(
    query: str,
    top_k: int = 5,
    candidate_pool: int = 20,
    first_stage: str = 'semantic',  # 'semantic' | 'hybrid'
) -> pd.DataFrame:
    """
    Full 2-stage pipeline:
      Stage 1: bi-encoder retrieval → candidate pool
      Stage 2: cross-encoder reranking → top-k final

    Args:
        query          : natural language query
        top_k          : final number of results
        candidate_pool : how many candidates to retrieve in stage 1
        first_stage    : which model to use for stage 1
    """
    # Stage 1 — fast retrieval
    if first_stage == 'semantic':
        candidates = search_semantic(query, top_k=candidate_pool)
    elif first_stage == 'hybrid':
        # Import from notebook 08 logic inline
        from collections import defaultdict
        bm25_scores = bm25.get_scores(query.lower().split())
        bm25_idx = np.argsort(bm25_scores)[::-1][:candidate_pool]
        bm25_ids = df.iloc[bm25_idx]['isbn13'].tolist()

        q_emb = bi_encoder.encode([BGE_PREFIX + query], normalize_embeddings=True)
        dense_res = collection.query(
            query_embeddings=q_emb.tolist(),
            n_results=candidate_pool,
            include=['metadatas', 'documents'],
        )
        dense_ids = dense_res['ids'][0]

        # RRF
        rrf_scores = defaultdict(float)
        for rank, doc_id in enumerate(bm25_ids):
            rrf_scores[doc_id] += 1 / (60 + rank + 1)
        for rank, doc_id in enumerate(dense_ids):
            rrf_scores[doc_id] += 1 / (60 + rank + 1)

        top_ids = [d for d, _ in sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)][:candidate_pool]
        candidates = df[df['isbn13'].isin(set(top_ids))].copy()
    else:
        raise ValueError(f'Unknown first_stage: {first_stage}')

    # Stage 2 — cross-encoder rerank
    return rerank_with_cross_encoder(query, candidates, top_k=top_k)


# Smoke test
t0 = time.time()
test_res = search_with_reranking(
    'a heartbreaking story about family secrets in rural America',
    top_k=5, candidate_pool=20,
)
print(f'Reranking smoke test ({time.time()-t0:.1f}s):')
test_res[['title', 'categories', 'rerank_score']]

Reranking smoke test (0.4s):


,title,categories,rerank_score
0,Always Time to Die,Governors,-2.057149
1,Every Cloak Rolled in Blood,Fiction,-4.087937
2,What Happened?,Fiction,-4.395047
3,Wolfsong,Paranormal fiction,-4.563152
4,Tribute,Fiction,-5.441372


## 10. Qualitative Analysis — Reranking Effect

In [6]:
DEMO_QUERY = 'a story where someone slowly loses their grip on reality'

print(f'Query: "{DEMO_QUERY}"\n')

print('--- Stage 1: BGE-small (before reranking) ---')
stage1 = search_semantic(DEMO_QUERY, top_k=10)
print(stage1[['title', 'categories', 'score']].head(5).to_string(index=False))

print('\n--- Stage 2: After cross-encoder reranking ---')
stage2 = search_with_reranking(DEMO_QUERY, top_k=5, candidate_pool=10)
print(stage2[['title', 'categories', 'rerank_score']].to_string(index=False))

print('\n--- Rank changes ---')
stage1_ranks = {row['title']: i+1 for i, row in stage1.head(10).iterrows()}
for i, row in stage2.iterrows():
    old_rank = stage1_ranks.get(row['title'], '?')
    new_rank = i + 1
    direction = '↑' if isinstance(old_rank, int) and old_rank > new_rank else ('↓' if isinstance(old_rank, int) and old_rank < new_rank else '=')
    print(f'  {direction} {row["title"][:45]:45s}  {old_rank} → {new_rank}')

Query: "a story where someone slowly loses their grip on reality"

--- Stage 1: BGE-small (before reranking) ---
                                   title     categories  score
      Losing Hope Signed Limited Edition        Fiction 0.6956
                     Unreliable Narrator        Fiction 0.6926
                       The Lies We Weave        Fiction 0.6907
                       The Shrinking Man      Body size 0.6829
Islands and Captivity in Popular Culture Social Science 0.6778

--- Stage 2: After cross-encoder reranking ---
                                             title categories  rerank_score
                             The Imaginary Husband    Fiction     -5.209736
                               Unreliable Narrator    Fiction     -6.682141
                                 The Lies We Weave    Fiction     -7.573313
The Death of Ivan Ilych by Leo Tolstoy Illustrated    Unknown     -8.546101
                                     The 13th Hour    Fiction     -9.321548

--- 